In [3]:
%%writefile worker_funciones.py
import pandas as pd
import numpy as np

def limpiar_y_transformar_bloque(df_bloque):
    """
    Función trabajadora que limpia, recodifica variables DANE y calcula
    indicadores de acceso y brecha digital sobre un bloque de filas.
    """
    df = df_bloque.copy()
    
    cols_binarias = ["P765S1", "P765S2", "P765S3", "P765S4", "P1085S1", "P1085S2", "P1085S3"]
    for col in cols_binarias:
        df[col] = df[col].fillna(0).apply(lambda x: 1 if x == 1 else 0)
        
    df["USA_INTERNET"] = np.where(df["P1084"].isin([1, 2, 3, 4]), 1, 0)
    
    df["USA_COMPUTADOR"] = np.where((df["P1910"].isin([1, 2, 3, 4])) | (df["P1911"].isin([1, 2, 3, 4])), 1, 0)
    
    df["USA_CELULAR"] = np.where(df["P1082"] == 1, 1, 0)
    
    mapa_frecuencia = {
        1: "Todos los días",
        2: "Al menos 1 vez/semana",
        3: "Al menos 1 vez/mes",
        4: "Menos de 1 vez/mes",
        5: "No utiliza internet"
    }
    df["FRECUENCIA_INTERNET"] = df["P1084"].map(mapa_frecuencia).fillna("Sin dato")
    
    mapa_razon = {
        1: "No lo necesita / No le interesa",
        2: "No sabe usarlo",
        3: "Es muy costoso",
        4: "No hay cobertura en la zona",
        5: "No tiene dispositivo",
        6: "Por seguridad / privacidad",
        7: "Falta de tiempo",
        8: "No tiene permiso / restricción",
        9: "Otra razón"
    }
    df["RAZON_NO_INTERNET"] = np.where(
        df["USA_INTERNET"] == 1,
        "Aplica (Sí usa internet)",
        df["P1929"].map(mapa_razon).fillna("Otra / No especifica")
    )
    
    # 7. Clasificación por cuartiles aproximados de Ingreso Per Cápita (COP)
    condiciones_ingreso = [
        df["PERCAPITA"] <= 400000,
        (df["PERCAPITA"] > 400000) & (df["PERCAPITA"] <= 750000),
        (df["PERCAPITA"] > 750000) & (df["PERCAPITA"] <= 1400000),
        df["PERCAPITA"] > 1400000
    ]
    etiquetas_ingreso = ["1. Bajo (<= $400k)", "2. Medio-Bajo ($400k-$750k)", "3. Medio-Alto ($750k-$1.4M)", "4. Alto (> $1.4M)"]
    df["NIVEL_INGRESO"] = np.select(condiciones_ingreso, etiquetas_ingreso, default="Sin dato")
    
    # 8. Puntaje de Conectividad Digital (0 a 4 puntos según acceso, internet hogar, computador y celular)
    df["PUNTAJE_CONECTIVIDAD"] = (
        df["USA_INTERNET"] +
        df["P1085S1"] +      # Internet en el hogar
        df["USA_COMPUTADOR"] +
        df["USA_CELULAR"]
    )
    
    return df

Overwriting worker_funciones.py


In [12]:
import os
import time
import numpy as np
import pandas as pd
import multiprocessing as mp
import importlib
import worker_funciones

importlib.reload(worker_funciones)

ruta_unificado = "../data/processed/dataset_brecha_unificado.csv"
df_raw = pd.read_csv(ruta_unificado, low_memory=False)
print(f"Dataset cargado: {len(df_raw):,} registros | Núcleos lógicos disponibles en CPU: {mp.cpu_count()}")

tiempos_seq = []
for i in range(3):
    t0 = time.perf_counter()
    df_seq = worker_funciones.limpiar_y_transformar_bloque(df_raw)
    t1 = time.perf_counter()
    tiempos_seq.append(t1 - t0)

t_secuencial = np.mean(tiempos_seq)
print(f"\n[Secuencial - 1 proceso] Tiempo promedio (3 corridas): {t_secuencial:.4f} segundos")

resultados_rendimiento = [
    {"Modo": "Secuencial (1 proceso)", "Procesos (P)": 1, "Tiempo Promedio (s)": t_secuencial, "Speedup": 1.0, "Eficiencia (%)": 100.0}
]

for num_procesos in [2, 4]:
    tiempos_par = []
    for i in range(3):
        t0 = time.perf_counter()
        bloques = np.array_split(df_raw, num_procesos)
        with mp.Pool(processes=num_procesos) as pool:
            resultados_bloques = pool.map(worker_funciones.limpiar_y_transformar_bloque, bloques)
        df_par = pd.concat(resultados_bloques, ignore_index=True)
        t1 = time.perf_counter()
        tiempos_par.append(t1 - t0)
        
    t_par_prom = np.mean(tiempos_par)
    speedup = t_secuencial / t_par_prom
    eficiencia = (speedup / num_procesos) * 100
    
    resultados_rendimiento.append({
        "Modo": f"Paralelo ({num_procesos} procesos)",
        "Procesos (P)": num_procesos,
        "Tiempo Promedio (s)": round(t_par_prom, 4),
        "Speedup": round(speedup, 4),
        "Eficiencia (%)": round(eficiencia, 2)
    })

df_metricas = pd.DataFrame(resultados_rendimiento)
df_metricas["Tiempo Promedio (s)"] = df_metricas["Tiempo Promedio (s)"].round(4)
print("\nTABLA DE RENDIMIENTO: SECUENCIAL VS. MULTIPROCESSING")
display(df_metricas)

df_metricas.to_csv("../evidencias/capturas_cpd/metricas_multiprocessing.csv", index=False)
ruta_limpio = "../data/processed/tic_limpio_2025.csv"
df_par.to_csv(ruta_limpio, index=False, encoding="utf-8")
print(f"\nDataset limpio exportado para Persona 2 en: {ruta_limpio}")
print(f"Filas: {len(df_par):,} | Columnas: {df_par.shape[1]} | Idéntico al secuencial: {df_seq.equals(df_par)}")

Dataset cargado: 221,043 registros | Núcleos lógicos disponibles en CPU: 12

[Secuencial - 1 proceso] Tiempo promedio (3 corridas): 3.9308 segundos


C:\Users\USUARIO\anaconda3\envs\BrechaDigital\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
C:\Users\USUARIO\anaconda3\envs\BrechaDigital\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
C:\Users\USUARIO\anaconda3\envs\BrechaDigital\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
C:\Users\USUARIO\anaconda3\envs\BrechaDigital\lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead


TABLA DE RENDIMIENTO: SECUENCIAL VS. MULTIPROCESSING


,Modo,Procesos (P),Tiempo Promedio (s),Speedup,Eficiencia (%)
0,Secuencial (1 proceso),1,3.9308,1.0000,100.00
1,Paralelo (2 procesos),2,8.8005,0.4467,22.33
2,Paralelo (4 procesos),4,6.0525,0.6495,16.24



Dataset limpio exportado para Persona 2 en: ../data/processed/tic_limpio_2025.csv
Filas: 221,043 | Columnas: 35 | Idéntico al secuencial: True


In [13]:
df_final = df_par.copy()

df_final["P1929"] = df_final["P1929"].fillna(0).astype(int)

cols_enteras = [
    "DIRECTORIO", "SECUENCIA_ENCUESTA", "SECUENCIA_P", "ORDEN",
    "P1_DEPARTAMENTO", "P1_MUNICIPIO", "REGION", "CLASE", "CANT_PERSONAS_HOGAR",
    "P1910", "P1911", "P1912", "P1084", "P1929", "P1082", "P804",
    "P765S1", "P765S2", "P765S3", "P765S4", "P1085S1", "P1085S2", "P1085S3",
    "USA_INTERNET", "USA_COMPUTADOR", "USA_CELULAR", "PUNTAJE_CONECTIVIDAD"
]
for c in cols_enteras:
    df_final[c] = df_final[c].astype(int)

orden_columnas = [
    "DIRECTORIO", "SECUENCIA_ENCUESTA", "SECUENCIA_P", "ORDEN", "FEX_C",
    "P1_DEPARTAMENTO", "NOMBRE_DEPARTAMENTO", "P1_MUNICIPIO", "REGION", "CLASE", "ZONA",
    "CANT_PERSONAS_HOGAR", "I_HOGAR", "PERCAPITA", "NIVEL_INGRESO",
    "USA_INTERNET", "FRECUENCIA_INTERNET", "RAZON_NO_INTERNET",
    "USA_COMPUTADOR", "USA_CELULAR", "PUNTAJE_CONECTIVIDAD",
    "P765S1", "P765S2", "P765S3", "P765S4",
    "P1085S1", "P1085S2", "P1085S3",
    "P1084", "P1929", "P1910", "P1911", "P1912", "P1082", "P804"
]
df_final = df_final[orden_columnas]

ruta_limpio = "../data/processed/tic_limpio_2025.csv"
df_final.to_csv(ruta_limpio, index=False, encoding="utf-8-sig")

print("AUDITORÍA DE CALIDAD DEL DATASET LIMPIO FINAL")
print(f"Filas: {df_final.shape[0]:,} | Columnas: {df_final.shape[1]}")
print(f"Total de valores nulos en TODO el dataset: {df_final.isna().sum().sum()}")
print("\nPrimeras 5 filas (columnas clave):")
display(df_final[["DIRECTORIO", "NOMBRE_DEPARTAMENTO", "ZONA", "NIVEL_INGRESO", "USA_INTERNET", "FRECUENCIA_INTERNET", "RAZON_NO_INTERNET", "PUNTAJE_CONECTIVIDAD"]].head(5))

AUDITORÍA DE CALIDAD DEL DATASET LIMPIO FINAL
Filas: 221,043 | Columnas: 35
Total de valores nulos en TODO el dataset: 0

Primeras 5 filas (columnas clave):


,DIRECTORIO,NOMBRE_DEPARTAMENTO,ZONA,NIVEL_INGRESO,USA_INTERNET,FRECUENCIA_INTERNET,RAZON_NO_INTERNET,PUNTAJE_CONECTIVIDAD
0,8369550,Bolívar,Urbano,2. Medio-Bajo ($400k-$750k),1,Todos los días,Aplica (Sí usa internet),4
1,8369550,Bolívar,Urbano,2. Medio-Bajo ($400k-$750k),1,Todos los días,Aplica (Sí usa internet),4
2,8369550,Bolívar,Urbano,2. Medio-Bajo ($400k-$750k),1,Todos los días,Aplica (Sí usa internet),4
3,8369550,Bolívar,Urbano,2. Medio-Bajo ($400k-$750k),1,Al menos 1 vez/semana,Aplica (Sí usa internet),3
4,8369551,Bolívar,Urbano,3. Medio-Alto ($750k-$1.4M),1,Todos los días,Aplica (Sí usa internet),3


In [10]:
%pip install openpyxl



   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import pandas as pd

ruta_divipola = "../data/docs_dane/DIVIPOLA_Municipios.xlsx"
df_div = pd.read_excel(ruta_divipola, sheet_name="Municipios", skiprows=10, dtype=str)

df_div.columns = ["COD_DEPTO", "NOM_DEPTO", "CODIGO_DIVIPOLA", "NOMBRE_CIUDAD_RAW", "TIPO_MPIO", "LONGITUD", "LATITUD", "NOTA"]

df_div_limpio = df_div[
    df_div["CODIGO_DIVIPOLA"].notna() & 
    df_div["CODIGO_DIVIPOLA"].str.strip().str.match(r"^\d{5}$")
].copy()

df_div_limpio["CODIGO_DIVIPOLA"] = df_div_limpio["CODIGO_DIVIPOLA"].str.strip()
df_div_limpio["NOMBRE_CIUDAD"] = df_div_limpio["NOMBRE_CIUDAD_RAW"].str.strip().str.title()
df_div_limpio["LATITUD"] = df_div_limpio["LATITUD"].astype(float)
df_div_limpio["LONGITUD"] = df_div_limpio["LONGITUD"].astype(float)

df_final = df_final.drop(columns=["NOMBRE_CIUDAD", "LATITUD", "LONGITUD"], errors="ignore")

df_final = pd.merge(
    df_final,
    df_div_limpio[["CODIGO_DIVIPOLA", "NOMBRE_CIUDAD", "LATITUD", "LONGITUD"]],
    on="CODIGO_DIVIPOLA",
    how="left"
)

cols = list(df_final.columns)
for col_mover in ["NOMBRE_CIUDAD", "LATITUD", "LONGITUD"]:
    cols.remove(col_mover)
idx_insert = cols.index("CODIGO_DIVIPOLA") + 1
cols = cols[:idx_insert] + ["NOMBRE_CIUDAD", "LATITUD", "LONGITUD"] + cols[idx_insert:]
df_final = df_final[cols]

df_final = df_final.sort_values(
    by=["NOMBRE_DEPARTAMENTO", "NOMBRE_CIUDAD", "DIRECTORIO", "ORDEN"]
).reset_index(drop=True)

ruta_spark_pbi = "../data/processed/tic_limpio_2025.csv"
ruta_excel = "../data/processed/tic_limpio_2025_excel.csv"

df_final.to_csv(ruta_spark_pbi, index=False, encoding="utf-8-sig")
df_final.to_csv(ruta_excel, sep=";", index=False, encoding="utf-8-sig")

print("--- CRUCE OFICIAL DIVIPOLA COMPLETADO ---")
print(f"Total registros: {len(df_final):,} | Columnas: {df_final.shape[1]}")
print(f"Municipios únicos con nombre oficial: {df_final['NOMBRE_CIUDAD'].nunique()}")
print(f"Municipios sin coincidencia (nulos):   {df_final['NOMBRE_CIUDAD'].isna().sum()}")
print(f"Total de nulos en TODO el dataset:    {df_final.isna().sum().sum()}")
print("\nVerificación del código 15820 (antes salía genérico):")
display(df_final[df_final["CODIGO_DIVIPOLA"] == "15820"][["NOMBRE_DEPARTAMENTO", "CODIGO_DIVIPOLA", "NOMBRE_CIUDAD", "LATITUD", "LONGITUD", "ZONA", "USA_INTERNET"]].head(3))

--- CRUCE OFICIAL DIVIPOLA COMPLETADO ---
Total registros: 221,043 | Columnas: 39
Municipios únicos con nombre oficial: 614
Municipios sin coincidencia (nulos):   0
Total de nulos en TODO el dataset:    0

Verificación del código 15820 (antes salía genérico):


,NOMBRE_DEPARTAMENTO,CODIGO_DIVIPOLA,NOMBRE_CIUDAD,LATITUD,LONGITUD,ZONA,USA_INTERNET
50105,Boyacá,15820,Tópaga,5.768201,-72.832245,Rural,1
50106,Boyacá,15820,Tópaga,5.768201,-72.832245,Rural,0
50107,Boyacá,15820,Tópaga,5.768201,-72.832245,Rural,1


In [14]:
df_final["CODIGO_DIVIPOLA"] = (
    df_final["P1_DEPARTAMENTO"].astype(str).str.zfill(2) +
    df_final["P1_MUNICIPIO"].astype(str).str.zfill(3)
)

mapa_capitales = {
    "05001": "Medellín", "08001": "Barranquilla", "11001": "Bogotá D.C.",
    "13001": "Cartagena", "15001": "Tunja", "17001": "Manizales",
    "18001": "Florencia", "19001": "Popayán", "20001": "Valledupar",
    "23001": "Montería", "25001": "Agua de Dios (Cund.)", "27001": "Quibdó",
    "41001": "Neiva", "44001": "Riohacha", "47001": "Santa Marta",
    "50001": "Villavicencio", "52001": "Pasto", "54001": "Cúcuta",
    "63001": "Armenia", "66001": "Pereira", "68001": "Bucaramanga",
    "70001": "Sincelejo", "73001": "Ibagué", "76001": "Cali",
    "81001": "Arauca", "85001": "Yopal", "86001": "Mocoa",
    "88001": "San Andrés", "91001": "Leticia", "94001": "Inírida",
    "95001": "San José del Guaviare", "97001": "Mitú", "99773": "Puerto Carreño"
}

df_final["NOMBRE_CIUDAD"] = df_final["CODIGO_DIVIPOLA"].map(mapa_capitales).fillna(
    "Municipio (" + df_final["CODIGO_DIVIPOLA"] + ") - " + df_final["NOMBRE_DEPARTAMENTO"]
)

df_final = df_final.sort_values(
    by=["NOMBRE_DEPARTAMENTO", "CODIGO_DIVIPOLA", "DIRECTORIO", "ORDEN"]
).reset_index(drop=True)

cols = list(df_final.columns)
for col_mover in ["CODIGO_DIVIPOLA", "NOMBRE_CIUDAD"]:
    cols.remove(col_mover)
idx_insert = cols.index("P1_MUNICIPIO") + 1
cols = cols[:idx_insert] + ["CODIGO_DIVIPOLA", "NOMBRE_CIUDAD"] + cols[idx_insert:]
df_final = df_final[cols]

ruta_spark_pbi = "../data/processed/tic_limpio_2025.csv"
ruta_excel = "../data/processed/tic_limpio_2025_excel.csv"

df_final.to_csv(ruta_spark_pbi, index=False, encoding="utf-8-sig")
df_final.to_csv(ruta_excel, sep=";", index=False, encoding="utf-8-sig")

print("DATASET ORDENADO GEOGRÁFICAMENTE POR DEPARTAMENTO Y CIUDAD")
print(f"Total registros: {len(df_final):,} | Columnas: {df_final.shape[1]} | Nulos totales: {df_final.isna().sum().sum()}")
print("\nMuestra de registros de ciudades capitales (ejemplo Boyacá / Tunja):")
display(df_final[df_final["NOMBRE_CIUDAD"] == "Tunja"][["NOMBRE_DEPARTAMENTO", "CODIGO_DIVIPOLA", "NOMBRE_CIUDAD", "ZONA", "USA_INTERNET", "PUNTAJE_CONECTIVIDAD"]].head(5))

DATASET ORDENADO GEOGRÁFICAMENTE POR DEPARTAMENTO Y CIUDAD
Total registros: 221,043 | Columnas: 37 | Nulos totales: 0

Muestra de registros de ciudades capitales (ejemplo Boyacá / Tunja):


,NOMBRE_DEPARTAMENTO,CODIGO_DIVIPOLA,NOMBRE_CIUDAD,ZONA,USA_INTERNET,PUNTAJE_CONECTIVIDAD
42889,Boyacá,15001,Tunja,Urbano,0,1
42890,Boyacá,15001,Tunja,Urbano,1,4
42891,Boyacá,15001,Tunja,Urbano,1,4
42892,Boyacá,15001,Tunja,Urbano,1,3
42893,Boyacá,15001,Tunja,Urbano,1,3
